# PlantCLEF 2015 LeafScan S-CNN Training

Strict workflow for reproducing the paper protocol. Use the Google Drive LeafScan archive first, then smoke-train, train both S-CNN stages, and evaluate the full species ranking.

## 1. Runtime Check

Select `Runtime -> Change runtime type -> GPU` before running training cells.

In [1]:
import torch

print('CUDA:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


CUDA: True
Device: NVIDIA L4


## 2. Clone Or Update Project

In [2]:
from pathlib import Path
import os
import shutil
import subprocess

PROJECT_DIR = Path('/content/diploma')
REPO_URL = 'https://github.com/robodanill/diploma.git'
BRANCH = 'robodanill/main'


def clone_project():
    os.chdir('/content')
    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)


def pull_project() -> bool:
    if not (PROJECT_DIR / '.git').exists():
        return False
    result = subprocess.run(['git', 'pull', '--ff-only'], cwd=PROJECT_DIR)
    return result.returncode == 0


if PROJECT_DIR.exists():
    print(f'Trying to update existing project: {PROJECT_DIR}')
    if not pull_project():
        print('Pull failed or project is not a git repository; cloning a fresh copy.')
        clone_project()
else:
    print(f'Project not found at {PROJECT_DIR}; cloning a fresh copy.')
    clone_project()

os.chdir(PROJECT_DIR)
subprocess.run(['python', '-m', 'pip', 'install', '-e', '.[ml]'], check=True)

commit = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], cwd=PROJECT_DIR, text=True).strip()
print(f'Project commit: {commit}')

import yaml
smoke_config = yaml.safe_load((PROJECT_DIR / 'configs/leafscan_smoke_training.yaml').read_text())
print('Smoke evaluation enabled:', smoke_config['evaluation']['enabled'])


Trying to update existing project: /content/diploma
Project commit: c520dcd
Smoke evaluation enabled: True


## 3. Mount Google Drive, Unpack LeafScan Dataset, And Create Split

Expected archive path: `/content/drive/MyDrive/PlantCLEF2015_leafscan_only.tar.gz`. Full training uses the official train metadata; `leafscan_metadata_split.csv` is created only for smoke validation.


In [4]:
from google.colab import drive

drive.mount('/content/drive')


ModuleNotFoundError: No module named 'google'

In [5]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma
ARCHIVE=/content/drive/MyDrive/PlantCLEF2015_leafscan_only.tar.gz
TEST_ARCHIVE=/content/drive/MyDrive/PlantCLEF2015_leafscan_test.tar.gz
if [ ! -f "$ARCHIVE" ]; then
  echo "Missing required LeafScan training archive: $ARCHIVE" >&2
  echo "Existing PlantCLEF archives in Google Drive:" >&2
  find /content/drive/MyDrive -maxdepth 1 -type f -iname 'PlantCLEF2015*.tar.gz' -printf '  %f\n' >&2 || true
  echo "Build it from the full PlantCLEF training package with:" >&2
  echo "  python scripts/build_plantclef_content_bundle.py --source-root /path/to/PlantCLEF2015/train --output $ARCHIVE --content LeafScan" >&2
  echo "Do not use PlantCLEF2015_leaf_only.tar.gz here; the paper protocol needs Content=LeafScan." >&2
  exit 2
fi
rm -rf data/plantclef2015
mkdir -p data/plantclef2015
tar -xzf "$ARCHIVE" -C data/plantclef2015
test -f data/plantclef2015/leafscan/metadata.csv
cp data/plantclef2015/leafscan/metadata.csv data/plantclef2015/leafscan_metadata.csv
if [ -f "$TEST_ARCHIVE" ]; then
  rm -rf data/plantclef2015/test_leafscan
  mkdir -p data/plantclef2015/test_leafscan
  tar -xzf "$TEST_ARCHIVE" -C data/plantclef2015/test_leafscan
  test -f data/plantclef2015/test_leafscan/leafscan/metadata.csv
  cp data/plantclef2015/test_leafscan/leafscan/metadata.csv data/plantclef2015/test_leafscan_metadata.csv
  echo "test leafscan archive extracted"
else
  echo "test leafscan archive not found at $TEST_ARCHIVE; run notebooks/plantclef_colab_test_data.ipynb first"
fi
plant-classifier-split-metadata \
  --metadata data/plantclef2015/leafscan_metadata.csv \
  --dataset-root data/plantclef2015/leafscan \
  --output data/plantclef2015/leafscan_metadata_split.csv \
  --train-ratio 0.70 \
  --val-ratio 0.15 \
  --test-ratio 0.15
wc -l data/plantclef2015/leafscan_metadata_split.csv
python - <<'PY2'
import csv
from collections import Counter
with open('data/plantclef2015/leafscan_metadata.csv', newline='', encoding='utf-8') as file:
    source_rows = list(csv.DictReader(file))
print('leafscan source rows:', len(source_rows))
print('leafscan source content:', Counter(row.get('content', '') for row in source_rows))
print('leafscan source genera:', len({row['genus'] for row in source_rows}))
print('leafscan source species:', len({row['species'] for row in source_rows}))
if len(source_rows) != 12605:
    raise RuntimeError(f'Expected 12605 PlantCLEF train LeafScan rows, got {len(source_rows)}')
with open('data/plantclef2015/leafscan_metadata_split.csv', newline='', encoding='utf-8') as file:
    rows = list(csv.DictReader(file))
print('smoke split:', Counter(row['split'] for row in rows))
PY2


CalledProcessError: Command 'b'set -euo pipefail\nexport PYTHONUNBUFFERED=1\ncd /content/diploma\nARCHIVE=/content/drive/MyDrive/PlantCLEF2015_leafscan_only.tar.gz\nTEST_ARCHIVE=/content/drive/MyDrive/PlantCLEF2015_leafscan_test.tar.gz\ntest -f "$ARCHIVE"\nrm -rf data/plantclef2015\nmkdir -p data/plantclef2015\ntar -xzf "$ARCHIVE" -C data/plantclef2015\ntest -f data/plantclef2015/leafscan/metadata.csv\ncp data/plantclef2015/leafscan/metadata.csv data/plantclef2015/leafscan_metadata.csv\nif [ -f "$TEST_ARCHIVE" ]; then\n  rm -rf data/plantclef2015/test_leafscan\n  mkdir -p data/plantclef2015/test_leafscan\n  tar -xzf "$TEST_ARCHIVE" -C data/plantclef2015/test_leafscan\n  test -f data/plantclef2015/test_leafscan/leafscan/metadata.csv\n  cp data/plantclef2015/test_leafscan/leafscan/metadata.csv data/plantclef2015/test_leafscan_metadata.csv\n  echo "test leafscan archive extracted"\nelse\n  echo "test leafscan archive not found at $TEST_ARCHIVE; run notebooks/plantclef_colab_test_data.ipyn

## 4. Smoke Train `S-CNN (A)` Genus

This is only a pipeline check on a small subset. It also runs genus retrieval evaluation after the epoch.


In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma
python -u -m plant_classifier.training.cli   --config configs/leafscan_smoke_training.yaml   --stage genus   --output checkpoints/smoke_scnn_genus_vgg16.pt


## 5. Smoke Evaluate `S-CNN (A)` On Validation Split


In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma
python -u -m plant_classifier.training.eval_genus_cli   --config configs/leafscan_smoke_training.yaml   --checkpoint checkpoints/smoke_scnn_genus_vgg16.pt   --max-species 40   --references-per-genus 2   --queries-per-genus 2   --top-k 5


## 6. Full VGG16 Train `S-CNN (A)` Genus

Trains the global-view genus model with `configs/leafscan_training.yaml` and saves `scnn_genus_vgg16.pt` plus `scnn_genus_vgg16_best.pt`.

In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma
python -u -m plant_classifier.training.cli   --config configs/leafscan_training.yaml   --stage genus   --output checkpoints/scnn_genus_vgg16.pt


## 7. Evaluate Full VGG16 `S-CNN (A)`

In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma
python -u -m plant_classifier.training.eval_genus_cli   --config configs/leafscan_training.yaml   --query-config configs/leafscan_test_vgg16.yaml   --checkpoint checkpoints/scnn_genus_vgg16_best.pt   --max-species 0   --references-per-genus 6   --reference-level species   --reference-split train   --top-k 5


## 8. Full VGG16 Train `S-CNN (B)` Species

Run this after VGG16 `S-CNN (A)` has a reasonable genus retrieval result.

In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma
python -u -m plant_classifier.training.cli   --config configs/leafscan_training.yaml   --stage species   --output checkpoints/scnn_species_vgg16.pt


## 9. Evaluate Full VGG16 Two-Stage Species Ranking

In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma
python -u -m plant_classifier.training.eval_species_cli \
  --config configs/leafscan_training.yaml \
  --query-config configs/leafscan_test_vgg16.yaml \
  --genus-checkpoint checkpoints/scnn_genus_vgg16_best.pt \
  --species-checkpoint checkpoints/scnn_species_vgg16_best.pt \
  --references-per-species 6 \
  --genus-candidates 30 \
  --reference-split train \
  --top-k 1 3 5


## 10. Smoke Train EfficientNet-B3

Checks both genus and species stages on a small subset before the full EfficientNet-B3 run.

In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma
python -u -m plant_classifier.training.cli   --config configs/smoke_training_efficientnet_b3.yaml   --stage genus   --output checkpoints/smoke_scnn_genus_efficientnet_b3.pt
python -u -m plant_classifier.training.cli   --config configs/smoke_training_efficientnet_b3.yaml   --stage species   --output checkpoints/smoke_scnn_species_efficientnet_b3.pt


## 10. Full EfficientNet-B3 Train `S-CNN (A)` Genus

In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma
python -u -m plant_classifier.training.cli   --config configs/leaf_training_efficientnet_b3.yaml   --stage genus   --output checkpoints/scnn_genus_efficientnet_b3.pt


## 11. Evaluate Full EfficientNet-B3 `S-CNN (A)`

In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma
python -u -m plant_classifier.training.eval_genus_cli   --config configs/leaf_training_efficientnet_b3.yaml   --query-config configs/leaf_test_efficientnet_b3.yaml   --checkpoint checkpoints/scnn_genus_efficientnet_b3_best.pt   --max-species 0   --references-per-genus 6   --reference-level species   --reference-split train   --top-k 5


## 12. Full EfficientNet-B3 Train `S-CNN (B)` Species

In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma
python -u -m plant_classifier.training.cli   --config configs/leaf_training_efficientnet_b3.yaml   --stage species   --output checkpoints/scnn_species_efficientnet_b3.pt


## 13. Smoke Train MobileNetV3-Large

Checks both genus and species stages on a small subset before the full MobileNetV3-Large run.

In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma
python -u -m plant_classifier.training.cli   --config configs/smoke_training_mobilenet_v3_large.yaml   --stage genus   --output checkpoints/smoke_scnn_genus_mobilenet_v3_large.pt
python -u -m plant_classifier.training.cli   --config configs/smoke_training_mobilenet_v3_large.yaml   --stage species   --output checkpoints/smoke_scnn_species_mobilenet_v3_large.pt


## 14. Full MobileNetV3-Large Train `S-CNN (A)` Genus

In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma
python -u -m plant_classifier.training.cli   --config configs/leaf_training_mobilenet_v3_large.yaml   --stage genus   --output checkpoints/scnn_genus_mobilenet_v3_large.pt


## 15. Evaluate Full MobileNetV3-Large `S-CNN (A)`

In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma
python -u -m plant_classifier.training.eval_genus_cli   --config configs/leaf_training_mobilenet_v3_large.yaml   --query-config configs/leaf_test_mobilenet_v3_large.yaml   --checkpoint checkpoints/scnn_genus_mobilenet_v3_large_best.pt   --max-species 0   --references-per-genus 6   --reference-level species   --reference-split train   --top-k 5


## 16. Full MobileNetV3-Large Train `S-CNN (B)` Species

In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma
python -u -m plant_classifier.training.cli   --config configs/leaf_training_mobilenet_v3_large.yaml   --stage species   --output checkpoints/scnn_species_mobilenet_v3_large.pt


## 17. Sync Checkpoints To Google Drive

Old Drive checkpoints are removed unless their name contains `_best`. Local `*_best` files are copied to Drive as regular checkpoint names, so Drive `*_best` files can mean best-across-all-runs.

In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma
python scripts/sync_checkpoints_to_drive.py \
  --source checkpoints \
  --dest /content/drive/MyDrive/diploma_checkpoints \
  --keep-token _best
ls -lh /content/drive/MyDrive/diploma_checkpoints


## 18. Build VGG16 Reference Index For Desktop Inference

In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma
python -u -m plant_classifier.training.build_index_cli \
  --config configs/leafscan_training.yaml \
  --genus-checkpoint checkpoints/scnn_genus_vgg16_best.pt \
  --species-checkpoint checkpoints/scnn_species_vgg16_best.pt \
  --output checkpoints/reference_index_leafscan_vgg16.pt
ls -lh checkpoints


## 19. Final Sync To Google Drive

In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma
python scripts/sync_checkpoints_to_drive.py \
  --source checkpoints \
  --dest /content/drive/MyDrive/diploma_checkpoints \
  --keep-token _best
ls -lh /content/drive/MyDrive/diploma_checkpoints
